# Chapter 09. 도구변수법 (Instrumental Variables)

## 학습 목표

- 내생성(endogeneity) 문제를 이해하고 도구변수로 해결하는 방법을 배운다
- 2단계 최소자승법(2SLS: Two-Stage Least Squares)을 수동 구현 및 statsmodels로 실행한다
- 1단계 F-통계량으로 약한 도구변수를 진단한다
- Wald 추정량의 개념과 계산을 이해한다
- LLM의 Structured Outputs로 도구변수를 자동 식별한다
- 실제 데이터(wage.csv)에서 2SLS를 적용한다

## 환경 설정

이 노트북은 OpenAI API를 사용하여 LLM 기반 도구변수 식별을 수행한다. 다음을 준비한다:
- `OPENAI_API_KEY` 환경 변수
- `statsmodels` (IV2SLS), `scipy`, `pandas`, `numpy`, `matplotlib`, `seaborn`

In [25]:
# 핵심 라이브러리 임포트
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

# 통계 및 IV 분석
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.sandbox.regression.gmm import IV2SLS

# OpenAI SDK
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional

# 환경 변수 로드
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 시각화 설정
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("라이브러리 임포트 완료.")

라이브러리 임포트 완료.


## 1. 내생성(Endogeneity) 문제

일반 최소자승법(OLS)은 처치 D와 오차항 ε이 독립이라고 가정한다: E[ε|D] = 0

그러나 **내생성**이 있으면 이 가정이 위배된다:
- **생략된 변수(Omitted variable)**: 결과 Y와 처치 D를 모두 영향하는 변수 U가 있으면, D와 ε이 상관된다
- 결과: OLS 추정량이 **편향(biased)**된다

이를 도구변수(Instrumental Variable, IV)로 해결할 수 있다.

In [26]:
# 시뮬레이션: 내생성으로 인한 OLS 편향

np.random.seed(42)

n = 500

# 교란변수 U: 결과 Y와 처치 D 모두에 영향
U = np.random.normal(loc=0, scale=1, size=n)

# 처치 D: U의 영향을 받음 (내생성)
D = 0.5 * U + np.random.normal(loc=0, scale=0.5, size=n)

# 결과 Y: τD + βU + ε
# 진정한 효과: τ = 2
tau_true = 2
beta_U = 1.5  # U의 영향
epsilon = np.random.normal(loc=0, scale=0.5, size=n)

Y = tau_true * D + beta_U * U + epsilon

# 데이터프레임
df_endogenous = pd.DataFrame({
    'Y': Y,
    'D': D,
    'U': U
})

print("내생성 시뮬레이션 데이터 생성:")
print(f"n = {n}")
print(f"진정한 효과: τ = {tau_true}")
print(f"\n공변량 U와 처치 D의 상관계수: {df_endogenous[['D', 'U']].corr().iloc[0, 1]:.4f}")
print(df_endogenous.head())

내생성 시뮬레이션 데이터 생성:
n = 500
진정한 효과: τ = 2

공변량 U와 처치 D의 상관계수: 0.6811
          Y         D         U
0  2.867641  0.711446  0.496714
1  2.026073  0.885576 -0.138264
2  0.250469 -0.375440  0.647689
3  4.047075  1.043000  1.523030
4 -0.886914 -0.442398 -0.234153


In [27]:
# OLS 회귀 (공변량 U 없음) - 편향된 추정

X_ols = sm.add_constant(df_endogenous[['D']])
ols_result = sm.OLS(df_endogenous['Y'], X_ols).fit()

tau_hat_ols = ols_result.params['D']
se_ols = ols_result.bse['D']

print("="*60)
print("[OLS 회귀] 공변량 U 없음 (내생성 존재)")
print("="*60)
print(f"회귀식: Y ~ D")
print(f"\nOLS 추정: tau_hat = {tau_hat_ols:.4f}")
print(f"표준오차: SE = {se_ols:.4f}")
print(f"진정한 값: τ = {tau_true:.4f}")
print(f"편향(Bias): {tau_hat_ols - tau_true:.4f}")
print(f"\n결론: OLS 추정이 진정한 값보다 {tau_hat_ols - tau_true:.4f}만큼 크다.")
print(f"이는 생략된 변수 U에 대한 편향이다.")

[OLS 회귀] 공변량 U 없음 (내생성 존재)
회귀식: Y ~ D

OLS 추정: tau_hat = 3.5154
표준오차: SE = 0.0772
진정한 값: τ = 2.0000
편향(Bias): 1.5154

결론: OLS 추정이 진정한 값보다 1.5154만큼 크다.
이는 생략된 변수 U에 대한 편향이다.


In [28]:
# OLS 회귀 (공변량 U 포함) - 올바른 추정

X_ols_adjusted = sm.add_constant(df_endogenous[['D', 'U']])
ols_adjusted_result = sm.OLS(df_endogenous['Y'], X_ols_adjusted).fit()

tau_hat_ols_adjusted = ols_adjusted_result.params['D']
se_ols_adjusted = ols_adjusted_result.bse['D']

print("="*60)
print("[OLS 회귀] 공변량 U 포함 (올바른 모형)")
print("="*60)
print(f"회귀식: Y ~ D + U")
print(f"\nOLS 추정: tau_hat = {tau_hat_ols_adjusted:.4f}")
print(f"표준오차: SE = {se_ols_adjusted:.4f}")
print(f"진정한 값: τ = {tau_true:.4f}")
print(f"편향(Bias): {tau_hat_ols_adjusted - tau_true:.4f}")
print(f"\n결론: U를 포함하면 편향이 거의 0이 된다.")

[OLS 회귀] 공변량 U 포함 (올바른 모형)
회귀식: Y ~ D + U

OLS 추정: tau_hat = 2.0745
표준오차: SE = 0.0463
진정한 값: τ = 2.0000
편향(Bias): 0.0745

결론: U를 포함하면 편향이 거의 0이 된다.


## 2. 도구변수(Instrumental Variable)의 직관

도구변수 Z는 다음 3가지 조건을 만족해야 한다:

1. **관련성(Relevance)**: Z는 처치 D와 관련 → Cov(Z, D) ≠ 0
2. **배제제약(Exclusion Restriction)**: Z는 Y에 오직 D를 통해서만 영향 → E[ε|Z] = 0
3. **독립성(Independence)**: Z는 생략된 변수 U와 독립 → Cov(Z, U) = 0

이 조건들이 만족되면, 도구변수를 이용한 2단계 최소자승법(2SLS)으로 τ를 일치 추정량(consistent estimator)으로 추정할 수 있다.

In [29]:
# 도구변수 시뮬레이션

np.random.seed(42)

n = 500

# 도구변수 Z: U와 독립
Z = np.random.normal(loc=0, scale=1, size=n)

# 교란변수 U: Y와 D를 모두 영향
U = np.random.normal(loc=0, scale=1, size=n)

# 처치 D: Z와 U의 영향을 받음
D_iv = 0.7 * Z + 0.4 * U + np.random.normal(loc=0, scale=0.5, size=n)

# 결과 Y: τD + βU + ε
tau_true = 2
beta_U = 1.5
epsilon = np.random.normal(loc=0, scale=0.5, size=n)

Y_iv = tau_true * D_iv + beta_U * U + epsilon

# 데이터프레임
df_iv = pd.DataFrame({
    'Y': Y_iv,
    'D': D_iv,
    'Z': Z,
    'U': U
})

print("도구변수 시뮬레이션 데이터 생성:")
print(f"n = {n}")
print(f"\n상관계수 행렬:")
print(df_iv[['Y', 'D', 'Z', 'U']].corr().round(4))
print(f"\n조건 확인:")
print(f"1. 관련성 (Relevance): Cov(Z, D) = {df_iv[['Z', 'D']].corr().iloc[0, 1]:.4f} (0이 아님)")
print(f"2. 배제제약 (Exclusion): Z는 Y에 직접 영향 없음")
print(f"3. 독립성 (Independence): Cov(Z, U) = {df_iv[['Z', 'U']].corr().iloc[0, 1]:.4f} (거의 0)")

도구변수 시뮬레이션 데이터 생성:
n = 500

상관계수 행렬:
        Y       D       Z       U
Y  1.0000  0.8667  0.4175  0.7847
D  0.8667  1.0000  0.6897  0.4147
Z  0.4175  0.6897  1.0000 -0.0757
U  0.7847  0.4147 -0.0757  1.0000

조건 확인:
1. 관련성 (Relevance): Cov(Z, D) = 0.6897 (0이 아님)
2. 배제제약 (Exclusion): Z는 Y에 직접 영향 없음
3. 독립성 (Independence): Cov(Z, U) = -0.0757 (거의 0)


## 3. 2SLS (Two-Stage Least Squares) 수동 구현

2SLS는 두 단계로 이루어진다:

**1단계**: 도구변수 Z로 처치 D를 예측
D_i = π₀ + π₁Z_i + v_i
예측값 D̂_i를 구한다.

**2단계**: 예측된 처치 D̂_i를 이용하여 Y를 회귀
Y_i = α + τD̂_i + ε_i
이렇게 구한 τ는 내생성으로부터 비편향된다.

In [30]:
# 2SLS 수동 구현 (NumPy)

# 1단계: D ~ Z
X_first_stage = sm.add_constant(df_iv[['Z']])
first_stage_result = sm.OLS(df_iv['D'], X_first_stage).fit()

# 예측된 처치 D_hat
D_hat = first_stage_result.predict(X_first_stage)

print("="*60)
print("[2SLS 1단계] D ~ Z")
print("="*60)
print(first_stage_result.summary().tables[1])
print(f"\nR² = {first_stage_result.rsquared:.4f}")
print(f"예측된 처치 D_hat의 처음 5개 값:")
print(D_hat.head())

[2SLS 1단계] D ~ Z
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0674      0.030      2.283      0.023       0.009       0.125
Z              0.6401      0.030     21.257      0.000       0.581       0.699

R² = 0.4757
예측된 처치 D_hat의 처음 5개 값:
0    0.385321
1   -0.021118
2    0.481957
3    1.042248
4   -0.082495
dtype: float64


In [31]:
# 2단계: Y ~ D_hat
X_second_stage = sm.add_constant(pd.DataFrame({'D_hat': D_hat}))
second_stage_result = sm.OLS(df_iv['Y'], X_second_stage).fit()

tau_hat_2sls = second_stage_result.params['D_hat']
se_2sls = second_stage_result.bse['D_hat']

print("="*60)
print("[2SLS 2단계] Y ~ D_hat")
print("="*60)
print(f"회귀식: Y ~ D_hat (1단계 예측값)")
print(f"\n2SLS 추정: tau_hat = {tau_hat_2sls:.4f}")
print(f"표준오차: SE = {se_2sls:.4f}")
print(f"진정한 값: τ = {tau_true:.4f}")
print(f"편향(Bias): {tau_hat_2sls - tau_true:.4f}")
print(f"\n결론: 2SLS가 OLS보다 진정한 값에 더 가깝다.")

[2SLS 2단계] Y ~ D_hat
회귀식: Y ~ D_hat (1단계 예측값)

2SLS 추정: tau_hat = 1.8735
표준오차: SE = 0.1827
진정한 값: τ = 2.0000
편향(Bias): -0.1265

결론: 2SLS가 OLS보다 진정한 값에 더 가깝다.


## 4. statsmodels IV2SLS

statsmodels는 IV2SLS 함수를 제공하여 자동으로 2SLS를 계산한다.
수식:
```
IV2SLS(endog, exog, instruments).fit()
```

- `endog`: 결과 변수 Y
- `exog`: 외생변수 (처치 D)
- `instruments`: 도구변수 Z

In [32]:
# statsmodels IV2SLS

# endog: Y
endog = df_iv[['Y']]

# exog: D (내생변수로 취급)
exog = sm.add_constant(df_iv[['D']])

# instruments: Z
instruments = sm.add_constant(df_iv[['Z']])

iv2sls_result = IV2SLS(endog, exog, instruments).fit()

tau_hat_iv2sls = iv2sls_result.params['D']
se_iv2sls = iv2sls_result.bse['D']

print("="*60)
print("[statsmodels IV2SLS]")
print("="*60)
print(iv2sls_result.summary())

[statsmodels IV2SLS]
                          IV2SLS Regression Results                           
Dep. Variable:                      Y   R-squared:                       0.683
Model:                         IV2SLS   Adj. R-squared:                  0.682
Method:                     Two Stage   F-statistic:                     273.7
                        Least Squares   Prob (F-statistic):           2.59e-49
Date:                Sat, 11 Apr 2026                                         
Time:                        23:34:11                                         
No. Observations:                 500                                         
Df Residuals:                     498                                         
Df Model:                           1                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0734      0.07

In [33]:
# 추정 결과 비교

comparison_df = pd.DataFrame({
    '추정 방법': ['진정한 값', 'OLS (편향)', '2SLS 수동', 'IV2SLS (statsmodels)'],
    '추정치': [tau_true, tau_hat_ols, tau_hat_2sls, tau_hat_iv2sls],
    '표준오차': [np.nan, se_ols, se_2sls, se_iv2sls],
    '편향': [0, tau_hat_ols - tau_true, tau_hat_2sls - tau_true, tau_hat_iv2sls - tau_true]
})

print("\n추정 결과 비교")
print(comparison_df.to_string(index=False))


추정 결과 비교
               추정 방법      추정치     표준오차        편향
               진정한 값 2.000000      NaN  0.000000
            OLS (편향) 3.515449 0.077220  1.515449
             2SLS 수동 1.873505 0.182728 -0.126495
IV2SLS (statsmodels) 1.873505 0.113246 -0.126495


## 5. 1단계 F-통계량 진단

약한 도구변수(Weak Instrument)는 다음 두 가지 문제를 야기한다:
1. 2SLS 추정량이 편향되고 분포가 정규분포에서 벗어남
2. 표준오차가 매우 커져 추정의 정밀도가 떨어짐

**진단 기준**: 1단계 F-통계량 > 10
- F < 10: 약한 도구변수 (Weak Instrument)
- F ≥ 10: 충분히 강한 도구변수

In [34]:
# 1단계 F-통계량 계산

# 1단계 회귀식: D ~ Z
X_fs = sm.add_constant(df_iv[['Z']])
fs_result = sm.OLS(df_iv['D'], X_fs).fit()

# F-통계량 추출
f_stat = fs_result.fvalue
f_pvalue = fs_result.f_pvalue

print("="*60)
print("[1단계 F-통계량 진단]")
print("="*60)
print(f"F-통계량: {f_stat:.4f}")
print(f"p값: {f_pvalue:.6f}")
print(f"\n판정:")
if f_stat > 10:
    print(f"강한 도구변수: F = {f_stat:.4f} > 10")
else:
    print(f"약한 도구변수: F = {f_stat:.4f} < 10")

[1단계 F-통계량 진단]
F-통계량: 451.8502
p값: 0.000000

판정:
강한 도구변수: F = 451.8502 > 10


In [35]:
# 약한 도구변수 시뮬레이션

np.random.seed(42)

n = 500

# 약한 도구변수 Z_weak: D와 약하게 관련
Z_weak = np.random.normal(loc=0, scale=1, size=n)
U_weak = np.random.normal(loc=0, scale=1, size=n)

# 처치 D: Z_weak의 영향이 작음
D_weak = 0.1 * Z_weak + 0.6 * U_weak + np.random.normal(loc=0, scale=0.5, size=n)

# 결과 Y
Y_weak = tau_true * D_weak + 1.5 * U_weak + np.random.normal(loc=0, scale=0.5, size=n)

df_weak = pd.DataFrame({
    'Y': Y_weak,
    'D': D_weak,
    'Z': Z_weak
})

# 약한 도구변수의 1단계 F-통계량
X_weak_fs = sm.add_constant(df_weak[['Z']])
weak_fs_result = sm.OLS(df_weak['D'], X_weak_fs).fit()
f_stat_weak = weak_fs_result.fvalue

print("="*60)
print("[약한 도구변수 진단]")
print("="*60)
print(f"1단계 F-통계량: {f_stat_weak:.4f}")
print(f"\n판정: F = {f_stat_weak:.4f} < 10 - 약한 도구변수")
print(f"\n결론: 이 도구변수는 신뢰할 수 없다.")

[약한 도구변수 진단]
1단계 F-통계량: 0.4689

판정: F = 0.4689 < 10 - 약한 도구변수

결론: 이 도구변수는 신뢰할 수 없다.


## 6. LLM 기반 도구변수 자동 식별

Pydantic과 OpenAI Structured Outputs를 이용하여 LLM이 자동으로 적절한 도구변수를 식별하도록 한다.

In [36]:
# Pydantic 모델: IV 식별 구조화 아웃풋

class IVIdentification(BaseModel):
    """도구변수 자동 식별 모델"""
    
    instrument_variable: str = Field(
        description="추천되는 도구변수 (예: 거리, 출생분기, 정책 변화)"
    )
    treatment: str = Field(
        description="처치 변수 (예: 교육 년수)"
    )
    outcome: str = Field(
        description="결과 변수 (예: 임금)"
    )
    confound: str = Field(
        description="잠재적 교란변수 (예: 능력)"
    )
    relevance_reason: str = Field(
        description="도구변수가 처치와 관련된 이유"
    )
    exclusion_reason: str = Field(
        description="도구변수가 배제제약을 만족하는 이유"
    )
    confidence_score: float = Field(
        ge=0, le=1,
        description="도구변수 타당성 신뢰도 (0-1)"
    )

print("IVIdentification 모델 정의 완료.")

IVIdentification 모델 정의 완료.


In [37]:
# LLM 기반 도구변수 식별

context = """
당신은 인과 추론 전문가다. 다음 상황에 대해 도구변수를 식별하고 그 타당성을 평가해야 한다.

상황: 교육이 임금에 미치는 인과 효과를 추정하고 싶다.
- 처치: 개인의 교육 년수
- 결과: 임금 (로그)
- 문제: 능력(ability) 같은 생략된 변수가 교육과 임금을 모두 영향할 수 있다 (내생성)

가능한 도구변수들:
1. 출생분기 (quarter of birth) - 학교 입학 시기 규제와 관련
2. 가까운 대학까지의 거리 (distance to nearest college) - 대학 입학에 영향
3. 부모의 교육 수준 - 능력과 상관될 수 있음 (배제제약 위배)

당신의 작업: 가장 적절한 도구변수를 선택하고 그 타당성을 평가하라.
"""


response = client.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": context}
    ],
    response_format=IVIdentification
)
# 구조화 아웃풋 추출
iv_result = response.choices[0].message.parsed

print("="*60)
print("[LLM 기반 도구변수 식별]")
print("="*60)
print(f"도구변수: {iv_result.instrument_variable}")
print(f"처치: {iv_result.treatment}")
print(f"결과: {iv_result.outcome}")
print(f"교란변수: {iv_result.confound}")
print(f"\n관련성: {iv_result.relevance_reason}")
print(f"\n배제제약: {iv_result.exclusion_reason}")
print(f"\n신뢰도: {iv_result.confidence_score:.2%}")

[LLM 기반 도구변수 식별]
도구변수: 출생분기 (quarter of birth)
처치: 개인의 교육 년수
결과: 임금 (로그)
교란변수: 능력 (ability)

관련성: 출생분기는 교육 입학 시기와 관련이 있어 교육 년수에 실질적으로 영향을 미칠 수 있다.

배제제약: 출생분기는 임금에 직접적인 영향을 미치지 않으며, 교육과 임금 사이의 내생성을 해결할 수 있는 합당한 도구변수이다.

신뢰도: 90.00%


## 7. 실제 데이터 분석: wage.csv

교육이 임금에 미치는 인과 효과를 IV로 추정한다:
- 결과: 로그 임금 (log wage)
- 처치: 교육 년수 (educ)
- 도구변수: 어머니 교육 수준 (meduc)
  - **관련성**: 부모의 교육 수준은 자녀의 교육과 강하게 상관된다
  - **배제제약**: 어머니 교육은 자녀의 임금에 직접 영향을 주지 않고, 자녀 교육을 통해서만 영향을 준다고 가정한다


In [38]:
# wage.csv 로드

data_path = './dataset/wage.csv'
df_wage = pd.read_csv(data_path)

# meduc(어머니 교육 수준)에 결측이 있으므로 제거한다
df_wage = df_wage.dropna(subset=['meduc']).reset_index(drop=True)

print(f'데이터 크기: {df_wage.shape}')
print(f'\n변수 목록:')
print(df_wage.columns.tolist())
print(f'\n기술통계:')
print(df_wage[['wage', 'educ', 'exper', 'meduc']].describe())
print(f'\n처음 5개 행:')
print(df_wage[['wage', 'educ', 'exper', 'meduc']].head())


데이터 크기: (857, 15)

변수 목록:
['wage', 'hours', 'IQ', 'educ', 'exper', 'tenure', 'age', 'married', 'black', 'south', 'urban', 'sibs', 'brthord', 'meduc', 'feduc']

기술통계:
              wage        educ       exper       meduc
count   857.000000  857.000000  857.000000  857.000000
mean    970.514586   13.575263   11.451575   10.682614
std     407.244209    2.197438    4.255596    2.849756
min     115.000000    9.000000    1.000000    0.000000
25%     675.000000   12.000000    8.000000    8.000000
50%     923.000000   13.000000   11.000000   12.000000
75%    1186.000000   16.000000   15.000000   12.000000
max    3078.000000   18.000000   22.000000   18.000000

처음 5개 행:
   wage  educ  exper  meduc
0   769    12     11    8.0
1   808    18     11   14.0
2   825    14     11   14.0
3   650    12     13   12.0
4   562    11     14    6.0


In [39]:
# OLS vs 2SLS 비교

# 변수 준비
y_wage = np.log(df_wage['wage'])
d_wage = df_wage['educ']
z_wage = df_wage['meduc']

# OLS: log(wage) ~ educ + exper
X_ols_wage = sm.add_constant(df_wage[['educ', 'exper']])
ols_wage_result = sm.OLS(y_wage, X_ols_wage).fit()

tau_hat_ols_wage = ols_wage_result.params['educ']
se_ols_wage = ols_wage_result.bse['educ']

print("="*60)
print("[OLS] log(wage) ~ educ + exper")
print("="*60)
print(f"교육의 임금 탄력성: {tau_hat_ols_wage:.4f}")
print(f"표준오차: {se_ols_wage:.4f}")
print(f"해석: 교육 1년 증가 -> 임금 {tau_hat_ols_wage*100:.2f}% 증가")
print(f"R2: {ols_wage_result.rsquared:.4f}")

[OLS] log(wage) ~ educ + exper
교육의 임금 탄력성: 0.0799
표준오차: 0.0068
해석: 교육 1년 증가 -> 임금 7.99% 증가
R2: 0.1414


In [40]:
# 2SLS: meduc를 도구변수로 사용

# IV2SLS (statsmodels)
endog_wage = y_wage
exog_wage = sm.add_constant(df_wage[['educ', 'exper']])
instruments_wage = sm.add_constant(df_wage[['meduc', 'exper']])

iv2sls_wage_result = IV2SLS(endog_wage, exog_wage, instruments_wage).fit()

tau_hat_iv_wage = iv2sls_wage_result.params['educ']
se_iv_wage = iv2sls_wage_result.bse['educ']

print('='*60)
print('[2SLS] log(wage) ~ educ + exper (IV: meduc)')
print('='*60)
print(f'교육의 임금 탄력성: {tau_hat_iv_wage:.4f}')
print(f'표준오차: {se_iv_wage:.4f}')
print(f'해석: 교육 1년 증가 -> 임금 {tau_hat_iv_wage*100:.2f}% 증가')
print(f'\n95% 신뢰구간: [{tau_hat_iv_wage - 1.96*se_iv_wage:.4f}, {tau_hat_iv_wage + 1.96*se_iv_wage:.4f}]')


[2SLS] log(wage) ~ educ + exper (IV: meduc)
교육의 임금 탄력성: 0.1497
표준오차: 0.0225
해석: 교육 1년 증가 -> 임금 14.97% 증가

95% 신뢰구간: [0.1057, 0.1938]


In [41]:
# 1단계 F-통계량 진단

X_first_wage = sm.add_constant(df_wage[['meduc', 'exper']])
first_stage_wage = sm.OLS(df_wage['educ'], X_first_wage).fit()

f_stat_wage = first_stage_wage.fvalue

print('='*60)
print('[1단계 F-통계량] educ ~ meduc + exper')
print('='*60)
print(f'F-통계량: {f_stat_wage:.4f}')
print(f'\n판정:')
if f_stat_wage > 10:
    print(f'강한 도구변수: F = {f_stat_wage:.4f} > 10')
else:
    print(f'약한 도구변수: F = {f_stat_wage:.4f} < 10')

print(f'\nEduc 회귀식:')
print(first_stage_wage.summary().tables[1])


[1단계 F-통계량] educ ~ meduc + exper
F-통계량: 172.2795

판정:
강한 도구변수: F = 172.2795 > 10

Educ 회귀식:
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         13.5576      0.330     41.143      0.000      12.911      14.204
meduc          0.2233      0.023      9.851      0.000       0.179       0.268
exper         -0.2068      0.015    -13.622      0.000      -0.237      -0.177


## 8. Wald 추정량

도구변수가 하나뿐인 경우, 2SLS 추정량은 **Wald 추정량**과 동일하다:

τ̂_IV = Cov(Y, Z) / Cov(D, Z)

이는 Reduced Form 회귀계수를 First Stage 회귀계수로 나눈 값이다.

In [42]:
# Wald 추정량 수동 계산

# Reduced Form: Y ~ Z
X_rf = sm.add_constant(df_iv[['Z']])
rf_result = sm.OLS(df_iv['Y'], X_rf).fit()
pi_rf = rf_result.params['Z']

# First Stage: D ~ Z
X_fs = sm.add_constant(df_iv[['Z']])
fs_result = sm.OLS(df_iv['D'], X_fs).fit()
pi_fs = fs_result.params['Z']

# Wald 추정량
tau_wald = pi_rf / pi_fs

print("="*60)
print("[Wald 추정량]")
print("="*60)
print(f"Reduced Form: Y ~ Z")
print(f"  계수: {pi_rf:.4f}")
print(f"\nFirst Stage: D ~ Z")
print(f"  계수: {pi_fs:.4f}")
print(f"\nWald 추정량: τ̂ = {pi_rf:.4f} / {pi_fs:.4f} = {tau_wald:.4f}")
print(f"IV2SLS 추정량: {tau_hat_iv2sls:.4f}")
print(f"\n차이: {abs(tau_wald - tau_hat_iv2sls):.6f} (반올림 오차)")

[Wald 추정량]
Reduced Form: Y ~ Z
  계수: 1.1992

First Stage: D ~ Z
  계수: 0.6401

Wald 추정량: τ̂ = 1.1992 / 0.6401 = 1.8735
IV2SLS 추정량: 1.8735

차이: 0.000000 (반올림 오차)
